# **CUDA Programming on NVIDIA GPUs**

# **Practical 2**

Again make sure the correct Runtime is being used, by clicking on the Runtime option at the top, then "Change runtime type", and selecting an appropriate GPU such as the T4.

Then verify with the instruction below the details of the GPU which is available to you.  

# **Answers**

## **1-2-3-** Reading
1 .Click on the link in the course webpage to the Google Colab notebook.**
2. Carefully follow the instructions in the notebook.
3. Read through the prac2.cu source file which you find in the notebook.

In [147]:
!nvidia-smi


Sat Jan 17 18:40:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [148]:
!wget https://people.maths.ox.ac.uk/gilesm/cuda/headers/helper_cuda.h
!wget https://people.maths.ox.ac.uk/gilesm/cuda/headers/helper_string.h


--2026-01-17 18:40:32--  https://people.maths.ox.ac.uk/gilesm/cuda/headers/helper_cuda.h
Resolving people.maths.ox.ac.uk (people.maths.ox.ac.uk)... 129.67.184.129, 2001:630:441:201::8143:b881
Connecting to people.maths.ox.ac.uk (people.maths.ox.ac.uk)|129.67.184.129|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 27832 (27K) [text/x-chdr]
Saving to: ‘helper_cuda.h.9’

helper_cuda.h.9     100%[===================>]  27.18K   158KB/s    in 0.2s    

2026-01-17 18:40:34 (158 KB/s) - ‘helper_cuda.h.9’ saved [27832/27832]

--2026-01-17 18:40:34--  https://people.maths.ox.ac.uk/gilesm/cuda/headers/helper_string.h
Resolving people.maths.ox.ac.uk (people.maths.ox.ac.uk)... 129.67.184.129, 2001:630:441:201::8143:b881
Connecting to people.maths.ox.ac.uk (people.maths.ox.ac.uk)|129.67.184.129|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 14875 (15K) [text/x-chdr]
Saving to: ‘helper_string.h.9’

helper_string.h.9   100%[===================>]  1

## 4-**Let's follow the directions in the notebook to compile and run the code and see the timings it gives.**


### **Monte Carlo Simulation with version 1**

In [149]:
%%writefile prac2.cu
////////////////////////////////////////////////////////////////////////
// GPU version of Monte Carlo algorithm using NVIDIA's CURAND library
////////////////////////////////////////////////////////////////////////

#include <stdlib.h>
#include <stdio.h>
#include <math.h>

#include <cuda.h>
#include <curand.h>

#include <helper_cuda.h>

////////////////////////////////////////////////////////////////////////
// CUDA global constants
////////////////////////////////////////////////////////////////////////

__constant__ int   N;
__constant__ float T, r, sigma, rho, alpha, dt, con1, con2;


////////////////////////////////////////////////////////////////////////
// kernel routine
////////////////////////////////////////////////////////////////////////


__global__ void pathcalc(float *d_z, float *d_v)
{
  float s1, s2, y1, y2, payoff;
  int   ind;

  // move array pointers to correct position
  // version 1
    ind = threadIdx.x + blockIdx.x*(2*N*blockDim.x);

    // version 2
    // ind = threadIdx.x*(2*N) + blockIdx.x*(2*N*blockDim.x);


    // path calculation

    s1 = 1.0f;
    s2 = 1.0f;

    for (int n=0; n<N; n++) {
      y1   = d_z[ind];
      // version 1
      ind += blockDim.x;      // shift pointer to next element
      // version 2
      // ind += 1;

      y2   = rho*y1 + alpha*d_z[ind];
      // version 1
      ind += blockDim.x;      // shift pointer to next element
      // version 2
      // ind += 1;

      s1 = s1*(con1 + con2*y1);
      s2 = s2*(con1 + con2*y2);
    }

  // put payoff value into device array

  payoff = 0.0f;
  if ( fabs(s1-1.0f)<0.1f && fabs(s2-1.0f)<0.1f ) payoff = exp(-r*T);

  d_v[threadIdx.x + blockIdx.x*blockDim.x] = payoff;
}


////////////////////////////////////////////////////////////////////////
// Main program
////////////////////////////////////////////////////////////////////////

int main(int argc, const char **argv){

  int     NPATH=9600000, h_N=100;
  float   h_T, h_r, h_sigma, h_rho, h_alpha, h_dt, h_con1, h_con2;
  float  *h_v, *d_v, *d_z;
  double  sum1, sum2;

  // initialise card

  findCudaDevice(argc, argv);

  // initialise CUDA timing

  float milli;
  cudaEvent_t start, stop;
  cudaEventCreate(&start);
  cudaEventCreate(&stop);

  // allocate memory on host and device

  h_v = (float *)malloc(sizeof(float)*NPATH);

  checkCudaErrors( cudaMalloc((void **)&d_v, sizeof(float)*NPATH) );
  checkCudaErrors( cudaMalloc((void **)&d_z, sizeof(float)*2*h_N*NPATH) );

  // define constants and transfer to GPU

  h_T     = 1.0f;
  h_r     = 0.05f;
  h_sigma = 0.1f;
  h_rho   = 0.5f;
  h_alpha = sqrt(1.0f-h_rho*h_rho);
  h_dt    = 1.0f/h_N;
  h_con1  = 1.0f + h_r*h_dt;
  h_con2  = sqrt(h_dt)*h_sigma;

  checkCudaErrors( cudaMemcpyToSymbol(N,    &h_N,    sizeof(h_N)) );
  checkCudaErrors( cudaMemcpyToSymbol(T,    &h_T,    sizeof(h_T)) );
  checkCudaErrors( cudaMemcpyToSymbol(r,    &h_r,    sizeof(h_r)) );
  checkCudaErrors( cudaMemcpyToSymbol(sigma,&h_sigma,sizeof(h_sigma)) );
  checkCudaErrors( cudaMemcpyToSymbol(rho,  &h_rho,  sizeof(h_rho)) );
  checkCudaErrors( cudaMemcpyToSymbol(alpha,&h_alpha,sizeof(h_alpha)) );
  checkCudaErrors( cudaMemcpyToSymbol(dt,   &h_dt,   sizeof(h_dt)) );
  checkCudaErrors( cudaMemcpyToSymbol(con1, &h_con1, sizeof(h_con1)) );
  checkCudaErrors( cudaMemcpyToSymbol(con2, &h_con2, sizeof(h_con2)) );

  // random number generation

  curandGenerator_t gen;
  checkCudaErrors( curandCreateGenerator(&gen, CURAND_RNG_PSEUDO_DEFAULT) );
  checkCudaErrors( curandSetPseudoRandomGeneratorSeed(gen, 1234ULL) );

  cudaEventRecord(start);
  checkCudaErrors( curandGenerateNormal(gen, d_z, 2*h_N*NPATH, 0.0f, 1.0f) );
  cudaEventRecord(stop);

  cudaEventSynchronize(stop);
  cudaEventElapsedTime(&milli, start, stop);

  printf("CURAND normal RNG  execution time (ms): %f,  samples/sec: %e \n",
          milli, 2.0*h_N*NPATH/(0.001*milli));

  // execute kernel and time it

  cudaEventRecord(start);
  pathcalc<<<NPATH/128, 128>>>(d_z, d_v);
  cudaEventRecord(stop);

  cudaEventSynchronize(stop);
  cudaEventElapsedTime(&milli, start, stop);

  getLastCudaError("pathcalc execution failed\n");

  printf("Monte Carlo kernel execution time (ms): %f \n",milli);

  // copy back results

  checkCudaErrors( cudaMemcpy(h_v, d_v, sizeof(float)*NPATH,
                   cudaMemcpyDefault) );

  // compute average

  sum1 = 0.0;
  sum2 = 0.0;
  for (int i=0; i<NPATH; i++) {
    sum1 += h_v[i];
    sum2 += h_v[i]*h_v[i];
  }

  printf("\nAverage value and standard deviation of error  = %13.8f %13.8f\n\n",
	 sum1/NPATH, sqrt((sum2/NPATH - (sum1/NPATH)*(sum1/NPATH))/NPATH) );

  // Tidy up library

  checkCudaErrors( curandDestroyGenerator(gen) );

  // Release memory and exit cleanly

  free(h_v);
  checkCudaErrors( cudaFree(d_v) );
  checkCudaErrors( cudaFree(d_z) );

  // CUDA exit -- needed to flush printf write buffer

  cudaDeviceReset();

}


Overwriting prac2.cu


### **Monte Carlo Simulation with version 2**

In [150]:
%%writefile prac2_version2.cu
////////////////////////////////////////////////////////////////////////
// GPU version of Monte Carlo algorithm using NVIDIA's CURAND library
////////////////////////////////////////////////////////////////////////

#include <stdlib.h>
#include <stdio.h>
#include <math.h>

#include <cuda.h>
#include <curand.h>

#include <helper_cuda.h>

////////////////////////////////////////////////////////////////////////
// CUDA global constants
////////////////////////////////////////////////////////////////////////

__constant__ int   N;
__constant__ float T, r, sigma, rho, alpha, dt, con1, con2;


////////////////////////////////////////////////////////////////////////
// kernel routine
////////////////////////////////////////////////////////////////////////


__global__ void pathcalc(float *d_z, float *d_v)
{
  float s1, s2, y1, y2, payoff;
  int   ind;

  // move array pointers to correct position
  // version 1
    //ind = threadIdx.x + blockIdx.x*(2*N*blockDim.x);

    // version 2
    ind = threadIdx.x*(2*N) + blockIdx.x*(2*N*blockDim.x);


    // path calculation

    s1 = 1.0f;
    s2 = 1.0f;

    for (int n=0; n<N; n++) {
      y1   = d_z[ind];
      // version 1
      //ind += blockDim.x;      // shift pointer to next element
      // version 2
      ind += 1;

      y2   = rho*y1 + alpha*d_z[ind];
      // version 1
      //ind += blockDim.x;      // shift pointer to next element
      // version 2
      ind += 1;

      s1 = s1*(con1 + con2*y1);
      s2 = s2*(con1 + con2*y2);
    }

  // put payoff value into device array

  payoff = 0.0f;
  if ( fabs(s1-1.0f)<0.1f && fabs(s2-1.0f)<0.1f ) payoff = exp(-r*T);

  d_v[threadIdx.x + blockIdx.x*blockDim.x] = payoff;
}


////////////////////////////////////////////////////////////////////////
// Main program
////////////////////////////////////////////////////////////////////////

int main(int argc, const char **argv){

  int     NPATH=9600000, h_N=100;
  float   h_T, h_r, h_sigma, h_rho, h_alpha, h_dt, h_con1, h_con2;
  float  *h_v, *d_v, *d_z;
  double  sum1, sum2;

  // initialise card

  findCudaDevice(argc, argv);

  // initialise CUDA timing

  float milli;
  cudaEvent_t start, stop;
  cudaEventCreate(&start);
  cudaEventCreate(&stop);

  // allocate memory on host and device

  h_v = (float *)malloc(sizeof(float)*NPATH);

  checkCudaErrors( cudaMalloc((void **)&d_v, sizeof(float)*NPATH) );
  checkCudaErrors( cudaMalloc((void **)&d_z, sizeof(float)*2*h_N*NPATH) );

  // define constants and transfer to GPU

  h_T     = 1.0f;
  h_r     = 0.05f;
  h_sigma = 0.1f;
  h_rho   = 0.5f;
  h_alpha = sqrt(1.0f-h_rho*h_rho);
  h_dt    = 1.0f/h_N;
  h_con1  = 1.0f + h_r*h_dt;
  h_con2  = sqrt(h_dt)*h_sigma;

  checkCudaErrors( cudaMemcpyToSymbol(N,    &h_N,    sizeof(h_N)) );
  checkCudaErrors( cudaMemcpyToSymbol(T,    &h_T,    sizeof(h_T)) );
  checkCudaErrors( cudaMemcpyToSymbol(r,    &h_r,    sizeof(h_r)) );
  checkCudaErrors( cudaMemcpyToSymbol(sigma,&h_sigma,sizeof(h_sigma)) );
  checkCudaErrors( cudaMemcpyToSymbol(rho,  &h_rho,  sizeof(h_rho)) );
  checkCudaErrors( cudaMemcpyToSymbol(alpha,&h_alpha,sizeof(h_alpha)) );
  checkCudaErrors( cudaMemcpyToSymbol(dt,   &h_dt,   sizeof(h_dt)) );
  checkCudaErrors( cudaMemcpyToSymbol(con1, &h_con1, sizeof(h_con1)) );
  checkCudaErrors( cudaMemcpyToSymbol(con2, &h_con2, sizeof(h_con2)) );

  // random number generation

  curandGenerator_t gen;
  checkCudaErrors( curandCreateGenerator(&gen, CURAND_RNG_PSEUDO_DEFAULT) );
  checkCudaErrors( curandSetPseudoRandomGeneratorSeed(gen, 1234ULL) );

  cudaEventRecord(start);
  checkCudaErrors( curandGenerateNormal(gen, d_z, 2*h_N*NPATH, 0.0f, 1.0f) );
  cudaEventRecord(stop);

  cudaEventSynchronize(stop);
  cudaEventElapsedTime(&milli, start, stop);

  printf("CURAND normal RNG  execution time (ms): %f,  samples/sec: %e \n",
          milli, 2.0*h_N*NPATH/(0.001*milli));

  // execute kernel and time it

  cudaEventRecord(start);
  pathcalc<<<NPATH/128, 128>>>(d_z, d_v);
  cudaEventRecord(stop);

  cudaEventSynchronize(stop);
  cudaEventElapsedTime(&milli, start, stop);

  getLastCudaError("pathcalc execution failed\n");

  printf("Monte Carlo kernel execution time (ms): %f \n",milli);

  // copy back results

  checkCudaErrors( cudaMemcpy(h_v, d_v, sizeof(float)*NPATH,
                   cudaMemcpyDefault) );

  // compute average

  sum1 = 0.0;
  sum2 = 0.0;
  for (int i=0; i<NPATH; i++) {
    sum1 += h_v[i];
    sum2 += h_v[i]*h_v[i];
  }

  printf("\nAverage value and standard deviation of error  = %13.8f %13.8f\n\n",
	 sum1/NPATH, sqrt((sum2/NPATH - (sum1/NPATH)*(sum1/NPATH))/NPATH) );

  // Tidy up library

  checkCudaErrors( curandDestroyGenerator(gen) );

  // Release memory and exit cleanly

  free(h_v);
  checkCudaErrors( cudaFree(d_v) );
  checkCudaErrors( cudaFree(d_z) );

  // CUDA exit -- needed to flush printf write buffer

  cudaDeviceReset();

}


Overwriting prac2_version2.cu


In [151]:
!nvcc prac2.cu -o prac2 -I. -lineinfo -arch=sm_70 --ptxas-options=-v --use_fast_math -lcudart -lcurand

ptxas info    : 0 bytes gmem, 36 bytes cmem[3]
ptxas info    : Compiling entry function '_Z8pathcalcPfS_' for 'sm_70'
ptxas info    : Function properties for _Z8pathcalcPfS_
    0 bytes stack frame, 0 bytes spill stores, 0 bytes spill loads
ptxas info    : Used 29 registers, 368 bytes cmem[0]


In [152]:
!./prac2

GPU Device 0: "Turing" with compute capability 7.5

CURAND normal RNG  execution time (ms): 105.562561,  samples/sec: 1.818827e+10 
Monte Carlo kernel execution time (ms): 29.532064 

Average value and standard deviation of error  =    0.41786269    0.00015237



**These timing results is around 29.499489 ms**

## 5-**Let's make a duplicate copy of the source code and uncomment the “Version 2” lines of code, in 3 places, and comment out the “Version 1” lines.**

In [153]:
!nvcc prac2.cu -o prac2 -I. -lineinfo -arch=sm_70 --ptxas-options=-v --use_fast_math -lcudart -lcurand

ptxas info    : 0 bytes gmem, 36 bytes cmem[3]
ptxas info    : Compiling entry function '_Z8pathcalcPfS_' for 'sm_70'
ptxas info    : Function properties for _Z8pathcalcPfS_
    0 bytes stack frame, 0 bytes spill stores, 0 bytes spill loads
ptxas info    : Used 29 registers, 368 bytes cmem[0]


In [154]:
!./prac2

GPU Device 0: "Turing" with compute capability 7.5

CURAND normal RNG  execution time (ms): 105.298752,  samples/sec: 1.823383e+10 
Monte Carlo kernel execution time (ms): 29.517344 

Average value and standard deviation of error  =    0.41786269    0.00015237



In [155]:
!nvcc prac2_version2.cu -o prac2_version2 -I. -lineinfo -arch=sm_70 --ptxas-options=-v --use_fast_math -lcudart -lcurand

ptxas info    : 0 bytes gmem, 36 bytes cmem[3]
ptxas info    : Compiling entry function '_Z8pathcalcPfS_' for 'sm_70'
ptxas info    : Function properties for _Z8pathcalcPfS_
    0 bytes stack frame, 0 bytes spill stores, 0 bytes spill loads
ptxas info    : Used 32 registers, 368 bytes cmem[0]


In [156]:
!./prac2_version2

GPU Device 0: "Turing" with compute capability 7.5

CURAND normal RNG  execution time (ms): 105.132225,  samples/sec: 1.826272e+10 
Monte Carlo kernel execution time (ms): 104.302429 

Average value and standard deviation of error  =    0.41793859    0.00015237



**The timing results show that version 1 runs in approximately 100.29 ms, while version 2 takes about 89 ms, making version 2 significantly slower.**

## **6- Let's explain why in Version 1 the 32 threads in a warp read in a consecutive block of 32 random numbers at the same time, but they don’t in Version 2.**

In [157]:
!nvcc prac2.cu -o prac2_device -I. -lineinfo -arch=sm_70 --ptxas-options=-v --use_fast_math -lcudart -lcurand

ptxas info    : 0 bytes gmem, 36 bytes cmem[3]
ptxas info    : Compiling entry function '_Z8pathcalcPfS_' for 'sm_70'
ptxas info    : Function properties for _Z8pathcalcPfS_
    0 bytes stack frame, 0 bytes spill stores, 0 bytes spill loads
ptxas info    : Used 29 registers, 368 bytes cmem[0]


In [158]:
!./prac2

GPU Device 0: "Turing" with compute capability 7.5

CURAND normal RNG  execution time (ms): 105.432449,  samples/sec: 1.821071e+10 
Monte Carlo kernel execution time (ms): 29.519424 

Average value and standard deviation of error  =    0.41786269    0.00015237



The performance difference between Version 1 and Version 2 comes solely from the way threads within the same warp access global memory. This can be understood by viewing the index layout as a matrix. In Version 1, if threads are arranged along the columns and time steps along the rows, then at a given instant all threads in a warp access consecutive memory indices. As a result, the GPU can combine the 32 memory loads into a single coalesced memory transaction, making efficient use of the available memory bandwidth. This corresponds to reading the matrix “row by row”, a pattern for which the GPU hardware is optimized. In contrast, in Version 2 the same matrix representation shows that, at a given instant, the threads access memory locations that are far apart from one another. The addresses are no longer contiguous, memory coalescing is lost, and the GPU must perform multiple separate memory transactions to satisfy what is logically a single load instruction for the warp.

## **7- Work out how much data is being loaded from device memory by the kernel, and based on the execution time determine the effective transfer rate in GB/s.**




In [159]:
%%writefile prac2_device.cu

////////////////////////////////////////////////////////////////////////
// GPU version of Monte Carlo algorithm using NVIDIA's CURAND library
////////////////////////////////////////////////////////////////////////

#include <stdlib.h>
#include <stdio.h>
#include <math.h>

#include <cuda.h>
#include <curand_kernel.h>

#include <helper_cuda.h>

////////////////////////////////////////////////////////////////////////
// CUDA global constants
////////////////////////////////////////////////////////////////////////

__constant__ int   N;
__constant__ float T, r, sigma, rho, alpha, dt, con1, con2;


////////////////////////////////////////////////////////////////////////
// kernel routines -- see sections 3.5, 3.6 in cuRAND documentation
////////////////////////////////////////////////////////////////////////

__global__ void RNG_init(curandState *state)
{
  // RNG initialisation with id-based skipahead
  int id = threadIdx.x + blockIdx.x*blockDim.x;
  curand_init(1234, id, 0, &state[id]);
}


__global__ void pathcalc(curandState *device_state, float *d_v,
                         int mpath, int NPATH)
{
  float s1, s2, y1, y2, payoff;

  int id = threadIdx.x + blockIdx.x*blockDim.x;
  curandState_t state = device_state[id];

  for(int m=0; m<mpath; m++) {
    s1 = 1.0f;
    s2 = 1.0f;

    for (int n=0; n<N; n++) {
      y1 = curand_normal(&state);
      y2 = rho*y1 + alpha*curand_normal(&state);

      s1 = s1*(con1 + con2*y1);
      s2 = s2*(con1 + con2*y2);
    }

    // put payoff value into device array

    payoff = 0.0f;
    if ( fabs(s1-1.0f)<0.1f && fabs(s2-1.0f)<0.1f ) payoff = exp(-r*T);

    int payoff_id = id + m*gridDim.x*blockDim.x;
    if (payoff_id < NPATH) d_v[payoff_id] = payoff;
  }
}


////////////////////////////////////////////////////////////////////////
// Main program
////////////////////////////////////////////////////////////////////////

int main(int argc, const char **argv){

  int     NPATH=9600000, h_N=100;
  float   h_T, h_r, h_sigma, h_rho, h_alpha, h_dt, h_con1, h_con2;
  float  *h_v, *d_v;
  double  sum1, sum2;
  curandState *state;

  // initialise card

  findCudaDevice(argc, argv);

  // initialise CUDA timing

  float milli;
  cudaEvent_t start, stop;
  cudaEventCreate(&start);
  cudaEventCreate(&stop);

  // allocate memory on host and device

  h_v = (float *)malloc(sizeof(float)*NPATH);
  checkCudaErrors( cudaMalloc((void **)&d_v, sizeof(float)*NPATH) );
  checkCudaErrors( cudaMalloc((void **)&state, sizeof(curandState)*NPATH) );

  // printf("size of curandState is %d bytes\n",sizeof(curandState));

  // define constants and transfer to GPU

  h_T     = 1.0f;
  h_r     = 0.05f;
  h_sigma = 0.1f;
  h_rho   = 0.5f;
  h_alpha = sqrt(1.0f-h_rho*h_rho);
  h_dt    = 1.0f/h_N;
  h_con1  = 1.0f + h_r*h_dt;
  h_con2  = sqrt(h_dt)*h_sigma;

  checkCudaErrors( cudaMemcpyToSymbol(N,    &h_N,    sizeof(h_N)) );
  checkCudaErrors( cudaMemcpyToSymbol(T,    &h_T,    sizeof(h_T)) );
  checkCudaErrors( cudaMemcpyToSymbol(r,    &h_r,    sizeof(h_r)) );
  checkCudaErrors( cudaMemcpyToSymbol(sigma,&h_sigma,sizeof(h_sigma)) );
  checkCudaErrors( cudaMemcpyToSymbol(rho,  &h_rho,  sizeof(h_rho)) );
  checkCudaErrors( cudaMemcpyToSymbol(alpha,&h_alpha,sizeof(h_alpha)) );
  checkCudaErrors( cudaMemcpyToSymbol(dt,   &h_dt,   sizeof(h_dt)) );
  checkCudaErrors( cudaMemcpyToSymbol(con1, &h_con1, sizeof(h_con1)) );
  checkCudaErrors( cudaMemcpyToSymbol(con2, &h_con2, sizeof(h_con2)) );

  // calculate theoretical occupancy -- see Pro Tip blog article:
  // https://developer.nvidia.com/blog/cuda-pro-tip-occupancy-api-simplifies-launch-configuration/

  int device;
  cudaDeviceProp props;
  cudaGetDevice(&device);
  cudaGetDeviceProperties(&props, device);

  int maxActiveBlocks, blockSize=128;
  cudaOccupancyMaxActiveBlocksPerMultiprocessor( &maxActiveBlocks,
                                                 pathcalc, blockSize, 0);
  printf("maxActiveBlocks/SM = %d \n",maxActiveBlocks);
  printf("number of SMs      = %d \n",props.multiProcessorCount);
  int blocks = maxActiveBlocks*props.multiProcessorCount;

  // execute kernels

  cudaEventRecord(start);
  RNG_init<<<blocks, 128>>>(state);
  cudaEventRecord(stop);

  cudaEventSynchronize(stop);
  cudaEventElapsedTime(&milli, start, stop);

  getLastCudaError("RNG_init execution failed\n");
  printf("RNG_init kernel execution time (ms): %f \n",milli);

  int paths_per_thread = (NPATH-1)/(128*blocks) + 1;
  cudaEventRecord(start);
  pathcalc<<<blocks, 128>>>(state,d_v,paths_per_thread,NPATH);
  cudaEventRecord(stop);

  cudaEventSynchronize(stop);
  cudaEventElapsedTime(&milli, start, stop);

  getLastCudaError("pathcalc execution failed\n");
  printf("pathcalc kernel execution time (ms): %f \n",milli);

  // copy back results

  checkCudaErrors( cudaMemcpy(h_v, d_v, sizeof(float)*NPATH,
                   cudaMemcpyDefault) );

  // compute average

  sum1 = 0.0;
  sum2 = 0.0;
  for (int i=0; i<NPATH; i++) {
    sum1 += h_v[i];
    sum2 += h_v[i]*h_v[i];
  }

  printf("\nAverage value and standard deviation of error  = %13.8f %13.8f\n\n",
	 sum1/NPATH, sqrt((sum2/NPATH - (sum1/NPATH)*(sum1/NPATH))/NPATH) );

  // Release memory and exit cleanly

  free(h_v);
  checkCudaErrors( cudaFree(d_v) );

  // CUDA exit -- needed to flush printf write buffer

  cudaDeviceReset();

}


Writing prac2_device.cu


In [160]:
!nvcc prac2_device.cu -o prac2_device -I. -lineinfo -arch=sm_70 --ptxas-options=-v --use_fast_math -lcudart -lcurand

ptxas info    : 218048 bytes gmem, 108 bytes cmem[3], 64 bytes cmem[4]
ptxas info    : Compiling entry function '_Z8pathcalcP17curandStateXORWOWPfii' for 'sm_70'
ptxas info    : Function properties for _Z8pathcalcP17curandStateXORWOWPfii
    0 bytes stack frame, 0 bytes spill stores, 0 bytes spill loads
ptxas info    : Used 29 registers, 376 bytes cmem[0]
ptxas info    : Compiling entry function '_Z8RNG_initP17curandStateXORWOW' for 'sm_70'
ptxas info    : Function properties for _Z8RNG_initP17curandStateXORWOW
    0 bytes stack frame, 0 bytes spill stores, 0 bytes spill loads
ptxas info    : Used 32 registers, 360 bytes cmem[0]


In [161]:
!./prac2_device

GPU Device 0: "Turing" with compute capability 7.5

maxActiveBlocks/SM = 8 
number of SMs      = 40 
RNG_init kernel execution time (ms): 1.651104 
pathcalc kernel execution time (ms): 23.686369 

Average value and standard deviation of error  =    0.41802440    0.00015237



Each thread reads 2 * N random numbers (y1 and y2) at each time step:

- N = 100 (number of time steps)  
- Size of a float = 4 bytes  
- Total number of threads = NPATH = 9,600,000  
- So each thread reads 2 * 100 = 200 floats  

**Total data calculation**:

$$Data\_loaded\_memory = 2 × number\_of\_steps × size\_of\_float × NPATH$$

$$Data\_loaded\_memory = 2 ×  100 ×  4 ×  9,600,000 = 7,680,000,000 \quad bytes$$
With:1 GB = 1 073 741 824 bytes

$$Data\_loaded\_memory ≈ 7.5 GB$$

**The Kernel execution time is**: 29.77 ms = 0.02977 s

$$
\text{Effective bandwidth} = \frac{7.5\ \text{GB}}{0.02977\ \text{s}} \approx 252\ \text{GB/s}
$$
**The Kernel execution time is (using device code)**: 23.70 ms = 0.02370 s

$$
\text{Effective bandwidth} = \frac{7.5\ \text{GB}}{0.02370\ \text{s}} \approx 316.45\ \text{GB/s}
$$

## **8- Write your own small program to compute the average value**

In [162]:
%%writefile prac2.cu
////////////////////////////////////////////////////////////////////////
// GPU version of Monte Carlo for a*z^2 + b*z + c
////////////////////////////////////////////////////////////////////////

#include <stdlib.h>
#include <stdio.h>
#include <math.h>

#include <cuda.h>
#include <curand_kernel.h> // needed for curandState
#include <helper_cuda.h>

////////////////////////////////////////////////////////////////////////
// CUDA global constants
///////////////////////////////////////////////////////////////////////

__constant__ int   N;
__constant__ float a, b, c;

///////////////////////////////////////////////////////////////////////
// kernel routine
////////////////////////////////////////////////////////////////////////

__global__ void pathcalc(float *d_v)
{
    float sum = 0.0f;
    int ind = threadIdx.x + blockIdx.x * blockDim.x;

    // initialize CURAND state per thread
    curandState state;
    curand_init(1234ULL + ind, 0, 0, &state);

    // each thread averages over N samples
    for (int n = 0; n < N; n++) {
        float z = curand_normal(&state); // We use for the standard normal
        sum += a*z*z + b*z + c;
    }

    d_v[ind] = sum / N;
}

////////////////////////////////////////////////////////////////////////
// Main program
////////////////////////////////////////////////////////////////////////

int main(int argc, const char **argv)
{
    int NPATH = 9600000;
    int h_N = 100;  // samples per thread
    float *h_v, *d_v;
    double sum1, sum2;

    // define constants
    float h_a = 1.0f, h_b = 2.0f, h_c = 3.0f;

    // initialise card
    findCudaDevice(argc, argv);

    // allocate memory
    h_v = (float*)malloc(sizeof(float)*NPATH);
    checkCudaErrors(cudaMalloc((void**)&d_v, sizeof(float)*NPATH));

    // copy constants to device
    checkCudaErrors(cudaMemcpyToSymbol(N, &h_N, sizeof(int)));
    checkCudaErrors(cudaMemcpyToSymbol(a, &h_a, sizeof(float)));
    checkCudaErrors(cudaMemcpyToSymbol(b, &h_b, sizeof(float)));
    checkCudaErrors(cudaMemcpyToSymbol(c, &h_c, sizeof(float)));

    // timing events
    float milli;
    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    // execute kernel
    cudaEventRecord(start);
    pathcalc<<<NPATH/128, 128>>>(d_v);
    cudaEventRecord(stop);

    cudaEventSynchronize(stop);
    cudaEventElapsedTime(&milli, start, stop);

    getLastCudaError("pathcalc execution failed\n");
    printf("Monte Carlo kernel execution time (ms): %f \n", milli);

    // copy back results
    checkCudaErrors(cudaMemcpy(h_v, d_v, sizeof(float)*NPATH, cudaMemcpyDeviceToHost));

    // compute average and std
    sum1 = 0.0;
    sum2 = 0.0;
    for (int i = 0; i < NPATH; i++) {
        sum1 += h_v[i];
        sum2 += h_v[i]*h_v[i];
    }

    printf("\nAverage value and standard deviation of error = %13.8f\n",sum1/NPATH);
    printf("Expected theoretical value = %f\n", 1.0f + 3.0f);

    // clean up
    free(h_v);
    checkCudaErrors(cudaFree(d_v));
    cudaDeviceReset();
}


Overwriting prac2.cu


In [163]:
!nvcc prac2.cu -o prac2 -I. -lineinfo -arch=sm_70 --ptxas-options=-v --use_fast_math -lcudart -lcurand

ptxas info    : 218048 bytes gmem, 88 bytes cmem[3], 64 bytes cmem[4]
ptxas info    : Compiling entry function '_Z8pathcalcPf' for 'sm_70'
ptxas info    : Function properties for _Z8pathcalcPf
    0 bytes stack frame, 0 bytes spill stores, 0 bytes spill loads
ptxas info    : Used 24 registers, 360 bytes cmem[0]


In [164]:
!./prac2

GPU Device 0: "Turing" with compute capability 7.5

Monte Carlo kernel execution time (ms): 8.636608 

Average value and standard deviation of error =    3.99978744
Expected theoretical value = 4.000000


We can notice the average value should be very close to a + c

In [ ]:
from google.colab import runtime
runtime.unassign()